# 02 · Notched pipe against the hybrid BEM, the full 3D BEM and the implicit COMSOL

**Shows.** The notched-pipe case of the paper solved with this code, compared with the
three other models, and the same case in a 3 m pipe.

**Problem.** L = 1 m, T(0,1) excited at z = L by a 40 kHz 5-cycle burst, 1 MPa,
t_max = 500 us. Notch on the outer surface: 2 mm deep, 90° wide, 10 mm long at
z = 0.51375–0.52375 m. Displacement on the outer surface at z = 0.75 m (incident wave
and echo) and z = 0.25 m (transmitted wave), θ = 0, 90, 180°.

**Compared with.** Hybrid BEM (reference), full 3D BEM and implicit COMSOL, from
`data/`. Errors: NRMSE of u_θ, u_r, u_z in % of the range of the reference u_θ; the
window error is u_θ in the scattered-wave window only (echo 210–360 μs, transmitted
230–380 μs), in % of its own range; the delay is by cross-correlation.

**Run time.** Reads results; produce them first with

```
julia -t 8 --project=. scripts/run_notched_pipe.jl 1 48 1.25    # ~10 min
julia -t 8 --project=. scripts/run_notched_pipe.jl 1 96 1.25    # ~30 min, mesh check (optional)
julia -t 8 --project=. scripts/run_notched_pipe.jl 3 48 1.25    # ~50 min, longer pipe (optional)
```

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # notebooks/Project.toml
using ExplicitGuidedWaves
using Plots, CSV, DataFrames, Printf

const PROJECT_DIR = pkgdir(ExplicitGuidedWaves)
const RESULTS_DIR = joinpath(PROJECT_DIR, "results")
const DATA_DIR = joinpath(PROJECT_DIR, "data")
const FIGURES_DIR = joinpath(RESULTS_DIR, "figures")
mkpath(FIGURES_DIR)
gr(fmt = :png)
println("Julia threads: ", Threads.nthreads())

In [ ]:
"One row per sensor of `compare_signal_sets`, as a table."
comparison_table(rows) = DataFrame([row for row in rows])   # comprehension: concrete row type

bem_times, bem_hybrid = reference_signals(:bem_hybrid, DATA_DIR)
_, bem_full = reference_signals(:bem_full_3d, DATA_DIR)
comsol_times, comsol = reference_signals(:comsol_implicit, DATA_DIR)

converged_case = NotchedPipeCase()            # L = 1 m, nc = 48, 1.25 mm
fem_times, fem = load_notched_signals(converged_case, RESULTS_DIR)
@printf("loaded %d sensors: BEM and COMSOL from data/, explicit FEM (%s) from results/; %d samples of 1 us\n",
        length(fem), case_name(converged_case), length(fem_times))

## 1. Against the hybrid BEM

In [ ]:
# a vector of pairs, not a Dict, so the table keeps this order
compared = ["full 3D BEM" => (bem_times, bem_full),
            "explicit FEM" => (fem_times, fem),
            "implicit COMSOL" => (comsol_times, comsol)]
tables = DataFrame[]
for (model, (times, signals)) in compared
    table = comparison_table(compare_signal_sets(times, signals, bem_times, bem_hybrid))
    table.model = fill(model, nrow(table))
    push!(tables, table)
end
against_bem = vcat(tables...)
CSV.write(joinpath(RESULTS_DIR, "notch_against_hybrid_bem.csv"), against_bem)
select(against_bem, :model, :sensor, :circumferential, :radial, :axial, :window, :delay_ns)

In [ ]:
"""
u_theta of one sensor in a time window (us) for the three models; each model is
(label, times in s, signals, colour, line style), signals as returned by `reference_signals`.
"""
function sensor_plot(sensor_name, window_us, models)
    # y range from the samples inside the window: Plots scales to all the data, and at
    # z = 0.75 m the incident wave (before 210 us) would flatten the echo
    peak_nm = 0.0
    for model in models
        inside = (1e6 .* model.times .>= window_us[1]) .& (1e6 .* model.times .<= window_us[2])
        peak_nm = max(peak_nm, maximum(abs, 1e9 .* model.signals[sensor_name].circumferential[inside]))
    end
    figure = plot(; xlims = window_us, ylims = (-1.15 * peak_nm, 1.15 * peak_nm),
                  xlabel = "t (μs)", ylabel = "u_θ (nm)", title = sensor_name)
    for model in models
        plot!(figure, 1e6 .* model.times, 1e9 .* model.signals[sensor_name].circumferential;
              label = model.label, color = model.color, linestyle = model.linestyle, linewidth = 1.5)
    end
    return figure
end

# drawn in this order: the hybrid BEM first, the other two on top
models = [(label = "hybrid BEM", times = bem_times, signals = bem_hybrid,
           color = MODEL_COLORS.hybrid_bem, linestyle = :solid),
          (label = "explicit FEM", times = fem_times, signals = fem,
           color = MODEL_COLORS.explicit_fem, linestyle = :dash),
          (label = "implicit COMSOL", times = comsol_times, signals = comsol,
           color = MODEL_COLORS.implicit_comsol, linestyle = :dot)]
figure = plot(sensor_plot("echo_theta0", (200, 380), models),
              sensor_plot("transmitted_theta0", (220, 400), models);
              layout = (2, 1), size = (750, 600))
savefig(figure, joinpath(FIGURES_DIR, "notch_three_models_theta0.png"))
figure

**Reading the result.** If this code and the BEM agree closely and the implicit COMSOL
differs from both by the same amount, with a delay, the difference is the numerical
dispersion of the implicit model, not an error of either of the other two.

## 2. Mesh convergence (optional runs)

In [ ]:
check_cases = [NotchedPipeCase(; n_circumferential = 96), NotchedPipeCase(; notch_axial_size = 0.000625)]
for case in check_cases
    if !isdir(joinpath(RESULTS_DIR, "notch", case_name(case)))
        println(case_name(case), ": not run (optional)")
        continue
    end
    times, signals = load_notched_signals(case, RESULTS_DIR)
    rows = compare_signal_sets(times, signals, fem_times, fem)
    worst = maximum(row.circumferential for row in rows)
    @printf("%s against %s: largest u_theta difference %.3f %%\n",
            case_name(case), case_name(converged_case), worst)
end

## 3. Longer pipe, L = 3 m (optional run)

Notch and sensors moved 1 m along z. Until end reflections arrive (385 μs of 1 m time),
the 3 m signals must equal the 1 m signals delayed by 1 m / c_T.

In [ ]:
long_case = NotchedPipeCase(; length = 3.0)
if isdir(joinpath(RESULTS_DIR, "notch", case_name(long_case)))
    long_times, long_signals = load_notched_signals(long_case, RESULTS_DIR)
    moved_times, moved = delayed_to_1m_time(long_case, long_times, long_signals)
    long_table = vcat(
        insertcols!(comparison_table(compare_signal_sets(moved_times, moved, fem_times, fem)),
                    1, :against => "explicit FEM, 1 m"),
        insertcols!(comparison_table(compare_signal_sets(moved_times, moved, bem_times, bem_hybrid)),
                    1, :against => "hybrid BEM, 1 m"),
    )
    select(long_table, :against, :sensor, :circumferential, :window, :delay_ns)
else
    println(case_name(long_case), ": not run (optional)")
end

## 4. Cost, same laptop

In [ ]:
fem_cost = DataFrame(CSV.File(joinpath(RESULTS_DIR, "notched_pipe.csv")))
select(fem_cost, :case, :n_dofs, :n_steps, :loop_s, :threads, :peak_memory_gb)

In [ ]:
# BEM rows of the v07 paper were measured with an earlier version of the BEM code; the
# current ones (cost mode, 23 Nov 2026) are those of the revised paper
costs = DataFrame(CSV.File(joinpath(DATA_DIR, "reference_costs.csv")))
earlier_bem_code(row) = startswith(row.method, "bem") && occursin("v07", row.source)
others = filter(row -> row.case == "notch" && !earlier_bem_code(row), costs)
select(others, :method, :length_m, :parallel, :time_s, :memory_gb, :source)